In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import time
import re

# Konfiguracja
BASE_URL = "https://lroc.im-ldi.com"
LIST_URL = "https://lroc.im-ldi.com/atlases/pits/list"
OUTPUT_DIR = "lunar_pits_png_dataset"

In [ ]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
def get_pit_links():
    print("Pobieranie listy obiektów...")
    try:
        response = requests.get(LIST_URL)
        soup = BeautifulSoup(response.text, 'html.parser')
        links = [a['href'] for a in soup.find_all('a', href=True)
                 if '/atlases/pits/' in a['href'] and a['href'] != '/atlases/pits/list']
        return list(set(links))
    except Exception as e:
        print(f"Błąd listy: {e}")
        return []

In [18]:
def download_image(pit_page_url):
    full_url = BASE_URL + pit_page_url
    pit_id = pit_page_url.split('/')[-1] # np. M125944637R

    try:
        response = requests.get(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        img_tags = soup.find_all('img')

        candidates = []

        for img in img_tags:
            src = img.get('src', '')
            src_lower = src.lower()

            # FILTRY:
            # 1. Musi zawierać ID zdjęcia (M...)
            # 2. Musi być formatem PNG (jak w Twoim przykładzie)
            # 3. WYKLUCZAMY wszystko co psuło zbiór poprzednio
            blacklist = ['topo', 'color', 'map', 'context', 'overlay', 'thumb', 'legend', 'hillshade']

            if pit_id in src and src_lower.endswith('.png'):
                if not any(word in src_lower for word in blacklist):
                    # Budujemy pełny URL
                    img_url = src if src.startswith('http') else BASE_URL + src
                    candidates.append(img_url)

        if candidates:
            # Jeśli jest kilka PNG, szukamy tego, który pasuje do Twojego wzorca (z podkreślnikami przed ID)
            # Przykład: Weiner_F_1_M125944637R.png
            target_url = candidates[0]
            for c in candidates:
                if "_" + pit_id in c: # Szuka wzorca [Coś]_ID.png
                    target_url = c
                    break

            print(f"Znaleziono właściwy plik PNG: {target_url.split('/')[-1]}")
            img_data = requests.get(target_url).content

            # Zapisujemy pod oryginalną nazwą z serwera, żeby zachować strukturę (np. Weiner_F_1_...)
            file_name = target_url.split('/')[-1]
            file_path = os.path.join(OUTPUT_DIR, file_name)

            with open(file_path, 'wb') as f:
                f.write(img_data)
            print(f"ZAPISANO: {file_name}")
        else:
            print(f"Nie znaleziono pasującego PNG dla {pit_id}")

    except Exception as e:
        print(f"Błąd przy {pit_id}: {e}")

Pobieranie listy obiektów...
Start pobierania 279 obiektów...
Znaleziono właściwy plik PNG: King_40_M1120456952R.png
ZAPISANO: King_40_M1120456952R.png
Znaleziono właściwy plik PNG: Hayn_1_M154705713R.png
ZAPISANO: Hayn_1_M154705713R.png
Znaleziono właściwy plik PNG: Copernicus_21_M155369774L.png
ZAPISANO: Copernicus_21_M155369774L.png
Znaleziono właściwy plik PNG: LROC_patch_alpha-bcddde05a14e141c980aa2f2525ba78b826a15843a289bf9fa543709e488e018.png
ZAPISANO: LROC_patch_alpha-bcddde05a14e141c980aa2f2525ba78b826a15843a289bf9fa543709e488e018.png
Znaleziono właściwy plik PNG: Kepler_2_M111843702R.png
ZAPISANO: Kepler_2_M111843702R.png
Znaleziono właściwy plik PNG: Tycho_8_M114024246R.png
ZAPISANO: Tycho_8_M114024246R.png
Znaleziono właściwy plik PNG: Lalande_12_M172975608L.png
ZAPISANO: Lalande_12_M172975608L.png
Znaleziono właściwy plik PNG: King_20h_M1131053889L.png
ZAPISANO: King_20h_M1131053889L.png
Znaleziono właściwy plik PNG: King_21_M1120456952R.png
ZAPISANO: King_21_M1120456952R.

In [ ]:
if __name__ == "__main__":
    links = get_pit_links()
    print(f"Start pobierania {len(links)} obiektów...")

    for link in links:
        download_image(link)
        time.sleep(0.3) # Szybkie pobieranie, mały delay

print("\nGotowe! Pliki są w folderze:", OUTPUT_DIR)